# Multimodal Fake News Detection
## DistilBERT + EfficientNetV2S + Metadata Fusion

**Creator:** Dhanush D | dhanushd1812@gmail.com | github.com/Drdhx

> Run cells top to bottom in VS Code with local Python kernel.

## Cell 1 - Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q transformers==4.41.0 timm==0.9.16 torchmetrics==1.3.2 \
    scikit-learn matplotlib seaborn pandas numpy Pillow tqdm torchvision accelerate
print('Install complete.')

## Cell 2 - Imports

In [ ]:
import os, warnings, random, time
from typing import List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import transforms

from transformers import (
    DistilBertTokenizer, DistilBertModel,
    get_linear_schedule_with_warmup
)
import timm

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, precision_recall_fscore_support,
    accuracy_score, f1_score
)
from sklearn.preprocessing import StandardScaler
from torchmetrics import Accuracy, F1Score
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
seed_everything()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('  Running on CPU')

## Cell 3 - Configuration

In [ ]:
class CFG:
    # Paths
    DATA_DIR        = r'C:\Users\Dhanush\OneDrive\Desktop\FND\multimodal'
    IMAGE_DIR       = r'C:\Users\Dhanush\OneDrive\Desktop\FND\multimodal\images'
    TRAIN_TSV       = r'C:\Users\Dhanush\OneDrive\Desktop\FND\multimodal\multimodal_train_small.tsv'
    VAL_TSV         = r'C:\Users\Dhanush\OneDrive\Desktop\FND\multimodal\multimodal_validate_small.tsv'
    TEST_TSV        = r'C:\Users\Dhanush\OneDrive\Desktop\FND\multimodal\multimodal_test_small.tsv'
    SAVE_DIR        = r'C:\Users\Dhanush\OneDrive\Desktop\FND\checkpoints'

    # HuggingFace model - uses local cache after first download
    # If download fails, set this to local folder path:
    # TEXT_MODEL_NAME = r'C:\Users\Dhanush\OneDrive\Desktop\FND\distilbert'
    TEXT_MODEL_NAME = 'distilbert-base-uncased'

    # Task
    NUM_CLASSES     = 2
    LABEL_COL       = '2_way_label'

    # Text
    MAX_LEN         = 128
    TEXT_DROPOUT    = 0.3

    # Image
    IMG_MODEL_NAME  = 'tf_efficientnetv2_s'
    IMG_SIZE        = 224
    IMG_DROPOUT     = 0.3

    # Metadata
    META_HIDDEN     = [128, 64]
    META_DIM        = 64
    META_DROPOUT    = 0.3

    # Fusion
    FUSION_DIM      = 256

    # Training
    BATCH_SIZE      = 16
    EPOCHS          = 10
    LR_BERT         = 2e-5
    LR_IMG          = 1e-4
    LR_HEAD         = 3e-4
    WEIGHT_DECAY    = 1e-2
    WARMUP_RATIO    = 0.1
    MAX_SAMPLES     = None
    NUM_WORKERS     = 0

os.makedirs(CFG.SAVE_DIR, exist_ok=True)
os.makedirs(CFG.IMAGE_DIR, exist_ok=True)

print('Config loaded.')
print(f'  Train : exists={os.path.exists(CFG.TRAIN_TSV)}')
print(f'  Val   : exists={os.path.exists(CFG.VAL_TSV)}')
print(f'  Test  : exists={os.path.exists(CFG.TEST_TSV)}')

## Cell 4 - Download DistilBERT (with retry on slow connections)

In [ ]:
import os

# Increase HuggingFace download timeout
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'

def load_distilbert_with_retry(model_name, max_attempts=5):
    for attempt in range(1, max_attempts + 1):
        try:
            print(f'Downloading {model_name} (attempt {attempt}/{max_attempts})...')
            model = DistilBertModel.from_pretrained(
                model_name,
                resume_download=True,
            )
            print('DistilBERT loaded successfully.')
            return model
        except Exception as e:
            print(f'  Attempt {attempt} failed: {type(e).__name__}')
            if attempt < max_attempts:
                wait = attempt * 5
                print(f'  Retrying in {wait}s...')
                time.sleep(wait)
            else:
                print('\n  All attempts failed.')
                print('  Manual fix: download model files from browser:')
                print('  https://huggingface.co/distilbert-base-uncased/resolve/main/model.safetensors')
                print('  https://huggingface.co/distilbert-base-uncased/resolve/main/config.json')
                print('  https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer.json')
                print('  https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json')
                print('  https://huggingface.co/distilbert-base-uncased/resolve/main/vocab.txt')
                print(r'  Save all to: C:\Users\Dhanush\OneDrive\Desktop\FND\distilbert\')
                print("  Then set CFG.TEXT_MODEL_NAME = r'C:\\Users\\Dhanush\\OneDrive\\Desktop\\FND\\distilbert'")
                raise

def load_tokenizer_with_retry(model_name, max_attempts=5):
    for attempt in range(1, max_attempts + 1):
        try:
            tok = DistilBertTokenizer.from_pretrained(
                model_name,
                resume_download=True,
            )
            print('Tokenizer loaded.')
            return tok
        except Exception as e:
            print(f'  Tokenizer attempt {attempt} failed: {type(e).__name__}')
            if attempt < max_attempts:
                time.sleep(attempt * 5)
            else:
                raise

# Pre-download tokenizer here so it is ready for DataLoader cell
tokenizer = load_tokenizer_with_retry(CFG.TEXT_MODEL_NAME)
print('All downloads ready.')

## Cell 5 - Load Data

In [ ]:
def load_split(path, label_col, max_samples=None):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Not found: {path}\nRun sample_dataset.py first.')
    df = pd.read_csv(path, sep='\t', on_bad_lines='skip', low_memory=False)
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    if 'clean_title' not in df.columns or label_col not in df.columns:
        print('Available columns:', df.columns.tolist())
        raise KeyError(f'Missing clean_title or {label_col}')
    df = df.dropna(subset=['clean_title', label_col])
    df[label_col] = df[label_col].astype(int)
    if max_samples:
        df = df.sample(min(max_samples, len(df)), random_state=SEED).reset_index(drop=True)
    return df

train_df = load_split(CFG.TRAIN_TSV, CFG.LABEL_COL, CFG.MAX_SAMPLES)
val_df   = load_split(CFG.VAL_TSV,   CFG.LABEL_COL)
test_df  = load_split(CFG.TEST_TSV,  CFG.LABEL_COL)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')
print('Columns:', train_df.columns.tolist())
train_df.head(3)

## Cell 6 - EDA Visualisations

In [ ]:
label_map = {0: 'Real', 1: 'Fake'}
sns.set_theme(style='whitegrid', font_scale=1.1)

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle('Fakeddit Dataset - EDA', fontsize=15, fontweight='bold')

ax1 = fig.add_subplot(gs[0, 0])
lc  = train_df[CFG.LABEL_COL].value_counts().sort_index()
bars = ax1.bar([label_map.get(i, str(i)) for i in lc.index],
               lc.values, color=['#2ecc71', '#e74c3c'])
ax1.bar_label(bars, fmt='%d')
ax1.set_title('Label Distribution'); ax1.set_xlabel('Class'); ax1.set_ylabel('Count')

ax2 = fig.add_subplot(gs[0, 1])
train_df['title_len'] = train_df['clean_title'].str.split().str.len()
ax2.hist(train_df['title_len'], bins=30, color='steelblue', edgecolor='white')
ax2.axvline(train_df['title_len'].median(), color='red', linestyle='--',
            label=f'Median: {train_df["title_len"].median():.0f}')
ax2.set_title('Title Word Length'); ax2.set_xlabel('Words'); ax2.legend()

ax3 = fig.add_subplot(gs[0, 2])
train_df['has_image'] = train_df['id'].apply(
    lambda pid: os.path.exists(os.path.join(CFG.IMAGE_DIR, f'{pid}.jpg')))
found   = int(train_df['has_image'].sum())
missing = len(train_df) - found
ax3.pie([found, missing], labels=[f'Found ({found})', f'Missing ({missing})'],
        colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%')
ax3.set_title('Image Availability')

ax4 = fig.add_subplot(gs[1, 0])
if 'upvote_ratio' in train_df.columns:
    for lbl, grp in train_df.groupby(CFG.LABEL_COL):
        ax4.hist(grp['upvote_ratio'].dropna(), bins=20, alpha=0.6,
                 label=label_map.get(lbl, str(lbl)))
    ax4.set_title('Upvote Ratio by Class'); ax4.legend()
else:
    ax4.text(0.5, 0.5, 'Not available', ha='center', va='center')
    ax4.set_title('Upvote Ratio')

ax5 = fig.add_subplot(gs[1, 1])
if 'score' in train_df.columns:
    clipped = train_df['score'].clip(upper=train_df['score'].quantile(0.95))
    ax5.hist(clipped, bins=30, color='coral', edgecolor='white')
    ax5.set_title('Post Score (clipped 95th pct)')
else:
    ax5.text(0.5, 0.5, 'Not available', ha='center', va='center')
    ax5.set_title('Score')

ax6 = fig.add_subplot(gs[1, 2])
if 'subreddit' in train_df.columns:
    train_df['subreddit'].value_counts().head(8).plot(kind='barh', ax=ax6, color='mediumpurple')
    ax6.set_title('Top Subreddits')
else:
    ax6.text(0.5, 0.5, 'Not available', ha='center', va='center')
    ax6.set_title('Subreddits')

plt.savefig(os.path.join(CFG.SAVE_DIR, 'eda.png'), dpi=120, bbox_inches='tight')
plt.show()
print('EDA saved.')

## Cell 7 - Metadata Feature Engineering

In [ ]:
META_NUMERIC = ['score', 'num_comments', 'upvote_ratio']
META_BINARY  = ['is_video', 'over_18', 'has_image']

def encode_subreddit(tr, va, te):
    if 'subreddit' not in tr.columns:
        return tr, va, te
    all_subs = pd.concat([tr['subreddit'], va['subreddit'],
                           te['subreddit']]).fillna('unknown').unique()
    sub2id = {s: i for i, s in enumerate(all_subs)}
    for df in [tr, va, te]:
        df['subreddit_enc'] = df['subreddit'].fillna('unknown').map(sub2id).fillna(0)
    return tr, va, te

train_df, val_df, test_df = encode_subreddit(train_df, val_df, test_df)

for df in [train_df, val_df, test_df]:
    df['title_len'] = df['clean_title'].str.split().str.len().fillna(0)
    if 'has_image' not in df.columns:
        df['has_image'] = df['id'].apply(
            lambda pid: os.path.exists(os.path.join(CFG.IMAGE_DIR, f'{pid}.jpg')))

META_COLS = []
for c in META_NUMERIC + META_BINARY + ['title_len']:
    if c in train_df.columns:
        META_COLS.append(c)
if 'subreddit_enc' in train_df.columns:
    META_COLS.append('subreddit_enc')

for df in [train_df, val_df, test_df]:
    for c in META_COLS:
        if c not in df.columns:
            df[c] = 0.0
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

scaler     = StandardScaler()
train_meta = scaler.fit_transform(train_df[META_COLS].astype(float))
val_meta   = scaler.transform(val_df[META_COLS].astype(float))
test_meta  = scaler.transform(test_df[META_COLS].astype(float))
META_INPUT_DIM = train_meta.shape[1]
print(f'Metadata dim: {META_INPUT_DIM}')
print(f'Features    : {META_COLS}')

## Cell 8 - Dataset and DataLoader

In [ ]:
TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
VAL_TRANSFORMS = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class FakedditDataset(Dataset):
    def __init__(self, df, meta_array, tokenizer, image_dir,
                 transform, label_col, max_len=128):
        self.df        = df.reset_index(drop=True)
        self.meta      = meta_array.astype(np.float32)
        self.tokenizer = tokenizer
        self.image_dir = image_dir
        self.transform = transform
        self.label_col = label_col
        self.max_len   = max_len
        self._blank    = Image.new('RGB', (CFG.IMG_SIZE, CFG.IMG_SIZE), (0, 0, 0))

    def __len__(self):
        return len(self.df)

    def _load_image(self, pid):
        path = os.path.join(self.image_dir, f'{pid}.jpg')
        try:
            return Image.open(path).convert('RGB')
        except Exception:
            return self._blank

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row['clean_title']),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        img = self._load_image(str(row.get('id', '')))
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'image':          self.transform(img),
            'metadata':       torch.tensor(self.meta[idx], dtype=torch.float32),
            'label':          torch.tensor(int(row[self.label_col]), dtype=torch.long),
        }

train_dataset = FakedditDataset(train_df, train_meta, tokenizer,
                                 CFG.IMAGE_DIR, TRAIN_TRANSFORMS, CFG.LABEL_COL, CFG.MAX_LEN)
val_dataset   = FakedditDataset(val_df,   val_meta,   tokenizer,
                                 CFG.IMAGE_DIR, VAL_TRANSFORMS,   CFG.LABEL_COL, CFG.MAX_LEN)
test_dataset  = FakedditDataset(test_df,  test_meta,  tokenizer,
                                 CFG.IMAGE_DIR, VAL_TRANSFORMS,   CFG.LABEL_COL, CFG.MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True,
                           num_workers=CFG.NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
val_loader   = DataLoader(val_dataset,   batch_size=CFG.BATCH_SIZE, shuffle=False,
                           num_workers=CFG.NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
test_loader  = DataLoader(test_dataset,  batch_size=CFG.BATCH_SIZE, shuffle=False,
                           num_workers=CFG.NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## Cell 9 - Model Architecture

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, model_name, out_dim, dropout=0.3):
        super().__init__()
        self.bert = load_distilbert_with_retry(model_name)
        self.proj = nn.Sequential(
            nn.LayerNorm(self.bert.config.hidden_size),
            nn.Dropout(dropout),
            nn.Linear(self.bert.config.hidden_size, out_dim),
            nn.GELU(),
        )
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.proj(out.last_hidden_state[:, 0, :])


class ImageEncoder(nn.Module):
    def __init__(self, model_name, out_dim, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=True, num_classes=0, global_pool='avg')
        self.proj = nn.Sequential(
            nn.LayerNorm(self.backbone.num_features),
            nn.Dropout(dropout),
            nn.Linear(self.backbone.num_features, out_dim),
            nn.GELU(),
        )
    def forward(self, x):
        return self.proj(self.backbone(x))


class MetadataEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dims, out_dim, dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, out_dim), nn.GELU()]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


class GatedFusion(nn.Module):
    def __init__(self, text_dim, img_dim, meta_dim, fusion_dim):
        super().__init__()
        self.text_proj = nn.Linear(text_dim,  fusion_dim)
        self.img_proj  = nn.Linear(img_dim,   fusion_dim)
        self.meta_proj = nn.Linear(meta_dim,  fusion_dim)
        self.gate = nn.Sequential(
            nn.Linear(text_dim + img_dim + meta_dim, 3),
            nn.Softmax(dim=-1)
        )
        self.norm = nn.LayerNorm(fusion_dim)
    def forward(self, t, v, m):
        gates = self.gate(torch.cat([t, v, m], dim=-1))
        fused = (gates[:, 0:1] * self.text_proj(t) +
                 gates[:, 1:2] * self.img_proj(v)  +
                 gates[:, 2:3] * self.meta_proj(m))
        return self.norm(fused)


class MultimodalFakeNewsDetector(nn.Module):
    def __init__(self, meta_input_dim, num_classes):
        super().__init__()
        F = CFG.FUSION_DIM
        self.text_encoder = TextEncoder(CFG.TEXT_MODEL_NAME, F, CFG.TEXT_DROPOUT)
        self.img_encoder  = ImageEncoder(CFG.IMG_MODEL_NAME, F, CFG.IMG_DROPOUT)
        self.meta_encoder = MetadataEncoder(
            meta_input_dim, CFG.META_HIDDEN, CFG.META_DIM, CFG.META_DROPOUT)
        self.fusion = GatedFusion(F, F, CFG.META_DIM, F)
        self.classifier = nn.Sequential(
            nn.Linear(F, 128), nn.LayerNorm(128), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(128, num_classes)
        )
    def forward(self, input_ids, attention_mask, image, metadata):
        t = self.text_encoder(input_ids, attention_mask)
        v = self.img_encoder(image)
        m = self.meta_encoder(metadata)
        return self.classifier(self.fusion(t, v, m))


model = MultimodalFakeNewsDetector(META_INPUT_DIM, CFG.NUM_CLASSES).to(DEVICE)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')

## Cell 10 - Training Setup

In [ ]:
label_counts_arr = np.array(
    [v for k, v in sorted(Counter(train_df[CFG.LABEL_COL]).items())])
class_weights = torch.tensor(
    1.0 / (label_counts_arr / label_counts_arr.sum()),
    dtype=torch.float32).to(DEVICE)
class_weights /= class_weights.sum()
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = AdamW([
    {'params': model.text_encoder.bert.parameters(),    'lr': CFG.LR_BERT},
    {'params': model.text_encoder.proj.parameters(),    'lr': CFG.LR_HEAD},
    {'params': model.img_encoder.backbone.parameters(), 'lr': CFG.LR_IMG},
    {'params': model.img_encoder.proj.parameters(),     'lr': CFG.LR_HEAD},
    {'params': model.meta_encoder.parameters(),         'lr': CFG.LR_HEAD},
    {'params': model.fusion.parameters(),               'lr': CFG.LR_HEAD},
    {'params': model.classifier.parameters(),           'lr': CFG.LR_HEAD},
], weight_decay=CFG.WEIGHT_DECAY)

total_steps  = len(train_loader) * CFG.EPOCHS
warmup_steps = int(total_steps * CFG.WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, warmup_steps, total_steps)

acc_metric = Accuracy(task='multiclass', num_classes=CFG.NUM_CLASSES).to(DEVICE)
f1_metric  = F1Score(task='multiclass', num_classes=CFG.NUM_CLASSES,
                     average='macro').to(DEVICE)
print('Training setup ready.')
print(f'Total steps: {total_steps} | Warmup: {warmup_steps}')

## Cell 11 - Training Loop

In [ ]:
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'train_f1':   [], 'val_f1':   []
}
best_val_f1 = 0.0
best_ckpt   = os.path.join(CFG.SAVE_DIR, 'best_model.pt')

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        pbar = tqdm(loader, leave=False, desc='Train' if train else 'Val  ')
        for batch in pbar:
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            imgs  = batch['image'].to(DEVICE)
            meta  = batch['metadata'].to(DEVICE)
            lbls  = batch['label'].to(DEVICE)
            logits = model(ids, mask, imgs, meta)
            loss   = criterion(logits, lbls)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            total_loss += loss.item() * len(lbls)
            all_preds.append(logits.argmax(-1))
            all_labels.append(lbls)
            pbar.set_postfix(loss=f'{loss.item():.4f}')
    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    avg_loss   = total_loss / len(loader.dataset)
    acc = acc_metric(all_preds, all_labels).item()
    f1  = f1_metric(all_preds, all_labels).item()
    acc_metric.reset(); f1_metric.reset()
    return avg_loss, acc, f1

print(f'Training {CFG.EPOCHS} epochs on {DEVICE}...')
print(f'Samples: {len(train_dataset)} train | Batch: {CFG.BATCH_SIZE}\n')

for epoch in range(1, CFG.EPOCHS + 1):
    t_loss, t_acc, t_f1 = run_epoch(train_loader, train=True)
    v_loss, v_acc, v_f1 = run_epoch(val_loader,   train=False)
    history['train_loss'].append(t_loss); history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc);   history['val_acc'].append(v_acc)
    history['train_f1'].append(t_f1);     history['val_f1'].append(v_f1)
    flag = ''
    if v_f1 > best_val_f1:
        best_val_f1 = v_f1
        torch.save(model.state_dict(), best_ckpt)
        flag = '  << best'
    print(f'Epoch {epoch:02d}/{CFG.EPOCHS} | '
          f'Train Loss:{t_loss:.4f} Acc:{t_acc:.4f} F1:{t_f1:.4f} | '
          f'Val Loss:{v_loss:.4f} Acc:{v_acc:.4f} F1:{v_f1:.4f}{flag}')

print(f'\nBest Val F1: {best_val_f1:.4f}')
print('Training complete.')

## Cell 12 - Training Curves

In [ ]:
epochs_range = range(1, CFG.EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training History', fontsize=13, fontweight='bold')
for ax, (tk, vk), title in zip(
    axes,
    [('train_loss','val_loss'), ('train_acc','val_acc'), ('train_f1','val_f1')],
    ['Loss', 'Accuracy', 'Macro F1']
):
    ax.plot(epochs_range, history[tk], 'o-', label='Train', color='royalblue')
    ax.plot(epochs_range, history[vk], 's-', label='Val',   color='tomato')
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG.SAVE_DIR, 'training_curves.png'), dpi=120)
plt.show()

## Cell 13 - Test Set Evaluation

In [ ]:
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()

all_preds_test, all_labels_test, all_probs_test = [], [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing'):
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        imgs  = batch['image'].to(DEVICE)
        meta  = batch['metadata'].to(DEVICE)
        lbls  = batch['label'].to(DEVICE)
        logits = model(ids, mask, imgs, meta)
        all_preds_test.append(logits.argmax(-1).cpu())
        all_labels_test.append(lbls.cpu())
        all_probs_test.append(F.softmax(logits, -1).cpu())

y_pred  = torch.cat(all_preds_test).numpy()
y_true  = torch.cat(all_labels_test).numpy()
y_probs = torch.cat(all_probs_test).numpy()

target_names = ['Real', 'Fake']
print('=' * 55)
print('  CLASSIFICATION REPORT - TEST SET')
print('=' * 55)
print(classification_report(y_true, y_pred,
      target_names=target_names, digits=4))
auc = roc_auc_score(y_true, y_probs[:, 1])
print(f'AUC-ROC: {auc:.4f}')

## Cell 14 - Result Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Test Set Results', fontsize=13, fontweight='bold')

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=axes[0, 0])
axes[0, 0].set_title('Confusion Matrix')
axes[0, 0].set_xlabel('Predicted'); axes[0, 0].set_ylabel('True')

fpr, tpr, _ = roc_curve(y_true, y_probs[:, 1])
axes[0, 1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC={auc:.4f}')
axes[0, 1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0, 1].set_title('ROC Curve')
axes[0, 1].set_xlabel('FPR'); axes[0, 1].set_ylabel('TPR')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

prec_arr, rec_arr, f1_arr, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=[0, 1])
x = np.arange(2)
axes[1, 0].bar(x - 0.2, prec_arr, 0.2, label='Precision', color='steelblue')
axes[1, 0].bar(x,        rec_arr,  0.2, label='Recall',    color='mediumseagreen')
axes[1, 0].bar(x + 0.2,  f1_arr,   0.2, label='F1',        color='tomato')
axes[1, 0].set_xticks(x); axes[1, 0].set_xticklabels(target_names)
axes[1, 0].set_ylim(0, 1.05); axes[1, 0].set_title('Per-Class Metrics')
axes[1, 0].legend(); axes[1, 0].grid(axis='y', alpha=0.3)

max_probs = y_probs.max(axis=1)
correct   = (y_pred == y_true)
axes[1, 1].hist(max_probs[correct],  bins=20, alpha=0.7,
                label='Correct',   color='mediumseagreen')
axes[1, 1].hist(max_probs[~correct], bins=20, alpha=0.7,
                label='Incorrect', color='tomato')
axes[1, 1].set_title('Confidence Distribution')
axes[1, 1].set_xlabel('Max Softmax Prob')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CFG.SAVE_DIR, 'results.png'), dpi=120)
plt.show()

## Cell 15 - Ablation Study

In [ ]:
def ablation_eval(loader, zero_text=False, zero_img=False, zero_meta=False):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False, desc='Ablation'):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            imgs = batch['image'].to(DEVICE)
            meta = batch['metadata'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            if zero_text:
                ids  = torch.zeros_like(ids)
                mask = torch.zeros_like(mask)
            if zero_img:  imgs = torch.zeros_like(imgs)
            if zero_meta: meta = torch.zeros_like(meta)
            all_p.append(model(ids, mask, imgs, meta).argmax(-1).cpu())
            all_l.append(lbls.cpu())
    p = torch.cat(all_p).numpy()
    l = torch.cat(all_l).numpy()
    return accuracy_score(l, p), f1_score(l, p, average='macro')

results = {
    'All modalities':  ablation_eval(test_loader),
    'No text':         ablation_eval(test_loader, zero_text=True),
    'No image':        ablation_eval(test_loader, zero_img=True),
    'No metadata':     ablation_eval(test_loader, zero_meta=True),
    'Text only':       ablation_eval(test_loader, zero_img=True, zero_meta=True),
    'Image only':      ablation_eval(test_loader, zero_text=True, zero_meta=True),
}
abl_df = pd.DataFrame(results, index=['Accuracy', 'Macro-F1']).T.round(4)
print(abl_df.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
x2 = np.arange(len(abl_df))
ax.bar(x2 - 0.2, abl_df['Accuracy'], 0.4, label='Accuracy', color='steelblue')
ax.bar(x2 + 0.2, abl_df['Macro-F1'], 0.4, label='Macro-F1',  color='tomato')
ax.set_xticks(x2)
ax.set_xticklabels(abl_df.index, rotation=15, ha='right')
ax.set_ylim(0, 1.05); ax.set_title('Ablation Study')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG.SAVE_DIR, 'ablation.png'), dpi=120)
plt.show()

## Cell 16 - Inference on New Samples

In [ ]:
def predict(title, image_path=None):
    model.eval()
    enc = tokenizer(
        title, max_length=CFG.MAX_LEN,
        padding='max_length', truncation=True, return_tensors='pt'
    )
    ids  = enc['input_ids'].to(DEVICE)
    mask = enc['attention_mask'].to(DEVICE)
    if image_path and os.path.exists(image_path):
        img = Image.open(image_path).convert('RGB')
    else:
        img = Image.new('RGB', (CFG.IMG_SIZE, CFG.IMG_SIZE), (0, 0, 0))
    img_t  = VAL_TRANSFORMS(img).unsqueeze(0).to(DEVICE)
    meta_t = torch.zeros(1, META_INPUT_DIM).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(
            model(ids, mask, img_t, meta_t), -1).squeeze().cpu().numpy()
    pred = int(probs.argmax())
    lmap = {0: 'Real', 1: 'Fake'}
    print(f'Title     : {title}')
    print(f'Prediction: {lmap[pred]}  (confidence: {probs.max():.3f})')
    print(f'Probs     : Real={probs[0]:.3f}  Fake={probs[1]:.3f}')
    print()

predict('Scientists discover water on Mars confirming life possibility')
predict('President signs new infrastructure bill worth $1 trillion')
predict('BREAKING: 5G towers secretly control your thoughts, experts warn')

## Cell 17 - Save Model

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'meta_cols':        META_COLS,
    'meta_input_dim':   META_INPUT_DIM,
    'best_val_f1':      best_val_f1,
    'history':          history,
}, os.path.join(CFG.SAVE_DIR, 'full_checkpoint.pt'))

tokenizer.save_pretrained(os.path.join(CFG.SAVE_DIR, 'tokenizer'))

print('=' * 50)
print('FINAL SUMMARY')
print('=' * 50)
print(f'Best Val F1 : {best_val_f1:.4f}')
print(f'Saved to    : {CFG.SAVE_DIR}')
print()
for f in sorted(os.listdir(CFG.SAVE_DIR)):
    fp = os.path.join(CFG.SAVE_DIR, f)
    if os.path.isfile(fp):
        print(f'  {f:<35} {os.path.getsize(fp)/1e6:6.1f} MB')

---
**Creator:** Dhanush D | dhanushd1812@gmail.com | github.com/Drdhx